# Task2: Support Ticket Classification & Prioritization System

An end-to-end NLP and machine learning project that:
- Classifies support tickets into business categories (Billing Issue, Technical Issue, Account Problem, General Query)
- Predicts ticket priority (High, Medium, Low) for automated triage

**Tech Stack:** Python · pandas · scikit-learn · TF-IDF · Logistic Regression · Naive Bayes · Random Forest

## 1. Setup & Imports

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

np.random.seed(42)
print('Libraries loaded successfully.')

## 2. Synthetic Dataset Generation

In [ ]:
samples = {
    ('Billing Issue',    'High'):   [
        'incorrect charge on my account',
        'refund not received after cancellation',
        'invoice amount does not match my plan',
        'duplicate payment was deducted',
        'I was charged for a service I did not use',
    ],
    ('Technical Issue',  'High'):   [
        'app keeps crashing on startup',
        'cannot login to my account',
        'server timeout error on every request',
        'payment gateway not responding',
        'data not syncing across devices',
    ],
    ('Account Problem',  'Medium'): [
        'need to reset my password',
        'account is locked after failed attempts',
        'profile update not saving',
        'two factor authentication not working',
        'unable to change email address',
    ],
    ('General Query',    'Low'):    [
        'how do I upgrade my plan',
        'what features are included in premium',
        'can I export my data',
        'how to use the dashboard analytics',
        'request for a new feature in the app',
    ],
}

rows = []
for (category, priority), texts in samples.items():
    for text in texts:
        rows.append({'ticket_text': text, 'category': category, 'priority': priority})

df = pd.DataFrame(rows)
df = pd.concat([df] * 40, ignore_index=True)
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
df.head(10)

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['category'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Ticket Category Distribution')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

df['priority'].value_counts().plot(kind='bar', ax=axes[1], color='tomato', edgecolor='white')
axes[1].set_title('Ticket Priority Distribution')
axes[1].set_xlabel('Priority')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=100)
plt.show()

## 4. Text Preprocessing

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['ticket_text'].apply(clean_text)
print('Sample cleaned text:')
print(df[['ticket_text', 'clean_text']].head(3).to_string())

## 5. Category Classification

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['category'],
    test_size=0.2, random_state=42, stratify=df['category']
)

cat_models = {
    'Logistic Regression': Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1, 2))), ('clf', LogisticRegression(max_iter=1000))]),
    'Naive Bayes':         Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1, 2))), ('clf', MultinomialNB())]),
    'Random Forest':       Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1, 2))), ('clf', RandomForestClassifier(n_estimators=100, random_state=42))]),
}

cat_results = {}
for name, model in cat_models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    cat_results[name] = acc
    print(f'\n{name} -- Accuracy: {acc:.4f}')
    print(classification_report(y_test, preds))

best_cat_name = max(cat_results, key=cat_results.get)
category_model = cat_models[best_cat_name]
print(f'\nBest category model: {best_cat_name} ({cat_results[best_cat_name]:.4f})')

## 6. Priority Prediction

In [ ]:
X2_train, X2_test, y2_train, y2_test = train_test_split(
    df['clean_text'], df['priority'],
    test_size=0.2, random_state=42, stratify=df['priority']
)

pri_models = {
    'Logistic Regression': Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1, 2))), ('clf', LogisticRegression(max_iter=1000))]),
    'Naive Bayes':         Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1, 2))), ('clf', MultinomialNB())]),
    'Random Forest':       Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1, 2))), ('clf', RandomForestClassifier(n_estimators=100, random_state=42))]),
}

pri_results = {}
for name, model in pri_models.items():
    model.fit(X2_train, y2_train)
    preds = model.predict(X2_test)
    acc = accuracy_score(y2_test, preds)
    pri_results[name] = acc
    print(f'\n{name} -- Accuracy: {acc:.4f}')
    print(classification_report(y2_test, preds))

best_pri_name = max(pri_results, key=pri_results.get)
priority_model = pri_models[best_pri_name]
print(f'\nBest priority model: {best_pri_name} ({pri_results[best_pri_name]:.4f})')

## 7. Model Comparison

In [ ]:
comparison_df = pd.DataFrame({
    'Model': list(cat_results.keys()),
    'Category Accuracy': list(cat_results.values()),
    'Priority Accuracy': list(pri_results.values()),
})

comparison_df.set_index('Model').plot(kind='bar', figsize=(10, 5), color=['steelblue', 'tomato'], edgecolor='white')
plt.title('Model Accuracy Comparison')
plt.ylabel('Accuracy')
plt.ylim(0, 1.1)
plt.xticks(rotation=20)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=100)
plt.show()
print(comparison_df.to_string(index=False))

## 8. Live Ticket Prediction

In [ ]:
def predict_ticket(ticket_text):
    t = clean_text(ticket_text)
    return {
        'ticket_text':        ticket_text,
        'predicted_category': category_model.predict([t])[0],
        'predicted_priority': priority_model.predict([t])[0],
    }

test_tickets = [
    'I was charged twice and need a refund immediately',
    'App crashes whenever I open it on my phone',
    'I forgot my password and cannot access my account',
    'Can you explain the difference between your plans',
]

pd.DataFrame([predict_ticket(t) for t in test_tickets])

# Task2: Support Ticket Classification & Prioritization System

This notebook builds an end-to-end NLP pipeline to classify support tickets by issue category and predict priority levels for triage automation.

In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

np.random.seed(42)

In [ ]:
billing = ["incorrect charge", "refund not received", "invoice mismatch"]
technical = ["app crashing", "cannot login", "server timeout"]
account = ["password reset", "account locked", "profile update issue"]
general = ["feature request", "how to use dashboard", "plan details"]

def build_rows(samples, category, priority):
    return [{"ticket_text": s, "category": category, "priority": priority} for s in samples]

rows = []
rows += build_rows(billing, "Billing Issue", "High")
rows += build_rows(technical, "Technical Issue", "High")
rows += build_rows(account, "Account Problem", "Medium")
rows += build_rows(general, "General Query", "Low")

df = pd.DataFrame(rows)
df = pd.concat([df] * 30, ignore_index=True)
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)
df.head()

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['ticket_text'].apply(clean_text)
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['category'], test_size=0.2, random_state=42, stratify=df['category']
)

In [ ]:
category_model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000))
])

category_model.fit(X_train, y_train)
preds = category_model.predict(X_test)
print('Category Accuracy:', round(accuracy_score(y_test, preds), 4))
print(classification_report(y_test, preds))

In [ ]:
X2_train, X2_test, y2_train, y2_test = train_test_split(
    df['clean_text'], df['priority'], test_size=0.2, random_state=42, stratify=df['priority']
)

priority_model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000))
])

priority_model.fit(X2_train, y2_train)
p_preds = priority_model.predict(X2_test)
print('Priority Accuracy:', round(accuracy_score(y2_test, p_preds), 4))
print(classification_report(y2_test, p_preds))

In [ ]:
def predict_ticket(ticket_text):
    t = clean_text(ticket_text)
    return {
        'ticket_text': ticket_text,
        'predicted_category': category_model.predict([t])[0],
        'predicted_priority': priority_model.predict([t])[0]
    }

predict_ticket('I was charged twice and need a refund immediately')